In [1]:
import pandas as pd
import numpy as np
import math
import networkx as nx

from collections import defaultdict
from sklearn.preprocessing import normalize
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import pairwise_distances, precision_score, recall_score, f1_score, accuracy_score

from joblib import Parallel, delayed

import igraph as ig

In [2]:
def minkowski_distance(x, y, p):
    distance = sum(abs(xi - yi) ** p for xi, yi in zip(x, y)) ** (1 / p)
    return distance

def find_maximal_cliques(adj):
    # 创建图
    G = nx.Graph(adj)
    # 找到所有极大团
    cliques = list(nx.find_cliques(G))
    return cliques

def ig_find_maximal_cliques(adj):
    # 使用 igraph 库来高效地找到极大团
    G = ig.Graph.Adjacency((adj > 0).tolist())
    cliques = G.maximal_cliques()
    return cliques

def matching_degree(G, h_G, y):    
    phi_G_y = sum(h_G[x] for x in G if minkowski_distance(y, X_train[x], 2) < delta)
    return phi_G_y

def argmaxphi(phi_values, labels):
    total_phi = sum(phi_values)
    if total_phi == 0:
        return ''
    
    label_aggregated_phi = defaultdict(float)
    for i, value in enumerate(phi_values):
        label_aggregated_phi[labels[i]] += value / total_phi
        
    return max(label_aggregated_phi, key=label_aggregated_phi.get)

In [3]:
data = pd.read_csv('sonar.csv')

y = data.iloc[:, -1].to_numpy()
scaler = MinMaxScaler()
X = scaler.fit_transform(data.iloc[:, :-1])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(data.shape)
n, m = X_train.shape
print(X_train[0])

(208, 61)
[0.12758112 0.15602229 0.30814717 0.18212078 0.04615775 0.10373555
 0.37635281 0.42271224 0.26625204 0.24209924 0.26229973 0.18859906
 0.25780913 0.4713829  0.72895978 0.77803786 0.78302767 0.59709091
 0.38985904 0.2302012  0.4886172  0.75145691 0.85122391 1.
 0.84190574 0.64720784 0.68179431 0.57214903 0.46956169 0.48876105
 0.40970027 0.13345316 0.12506563 0.03985162 0.38293955 0.71239919
 0.76186311 0.32504939 0.02298124 0.28028322 0.4447018  0.46617827
 0.31553084 0.25766555 0.32797839 0.25863961 0.35494386 0.50943396
 0.68955073 0.48242424 0.14243028 0.12125535 0.07272727 0.30116959
 0.05442177 0.13589744 0.24715909 0.12356979 0.184573   0.04157044]


In [104]:
data = pd.read_csv('sonar.csv')

y = data.iloc[:, -1].to_numpy()
scaler = MinMaxScaler()
X = scaler.fit_transform(data.iloc[:, :-1])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

print('Neighborhood')
print('# of test:', len(y_test))

delta = 0.15 * math.sqrt(m)
alpha = 0.7

print('delta: ', delta, 'alpha: ', alpha)


dis_arr = pairwise_distances(X_train, metric='euclidean')

# 初始化结果列表
neighborhoods = []
neighborhoods_label = []
granules = []

# 并行处理每个训练样本
def process_training_sample(i):
    # 找到与样本 i 距离小于 delta 的所有样本索引
    neighbors = np.where(dis_arr[i] < delta)[0]
    temp_label = y_train[neighbors]
    count = pd.Series(temp_label).value_counts()
    if not count[count > alpha * len(neighbors)].empty:
        label = count[count > alpha * len(neighbors)].index.tolist()[0]
        return i, neighbors, label
    else:
        return None

# 使用多线程并行处理
results = Parallel(n_jobs=-1)(delayed(process_training_sample)(i) for i in range(n))

# 过滤结果
for res in results:
    if res is not None:
        i, neighbors, label = res
        neighborhoods.append(i)
        granules.append(neighbors)
        neighborhoods_label.append(label)

print('Total: ', n, 'Filtered: ', len(neighborhoods))

acc = 0
dec = 0

# 将 neighborhoods 转换为 NumPy 数组以便于索引
neighborhoods = np.array(neighborhoods)

# 初始化变量以收集预测结果和真实标签
y_true = []
y_pred = []

# 并行处理测试样本
def process_test_sample(i, t):
    # 计算测试样本与所有训练样本的距离
    dis_to_train = np.linalg.norm(X_train - t, axis=1)
    # 找到距离小于 delta 的训练样本索引
    close_indices = neighborhoods[dis_to_train[neighborhoods] < delta]
    mc_labels = y_train[close_indices].tolist()
    if len(mc_labels)>0:
        # 使用多数投票法确定预测标签
        predicted_label = pd.Series(mc_labels).mode()[0]
        return y_test[i], predicted_label
    else:
        return y_test[i], None  # 无法预测时，返回 None

# 使用多线程并行处理测试样本
test_results = Parallel(n_jobs=-1)(delayed(process_test_sample)(i, t) for i, t in enumerate(X_test))

# 收集预测结果和真实标签
for true_label, pred_label in test_results:
    y_true.append(true_label)
    y_pred.append(pred_label)

# 过滤无法预测的样本
y_true_filtered = []
y_pred_filtered = []
for true_label, pred_label in zip(y_true, y_pred):
    if pred_label is not None:
        y_true_filtered.append(true_label)
        y_pred_filtered.append(pred_label)

# 计算评价指标
if y_pred_filtered:
    coverage = len(y_pred_filtered) / len(y_test)
    accuracy = accuracy_score(y_true_filtered, y_pred_filtered)
    precision = precision_score(y_true_filtered, y_pred_filtered, average='weighted', zero_division=0)
    recall = accuracy * coverage
    f1 = 2 * (precision * recall) / (precision + recall)
    print(f'Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1-score: {f1:.4f}, Coverage: {coverage:.4f}')
else:
    print('No predictions were made.')

Neighborhood
# of test: 42
delta:  1.161895003862225 alpha:  0.7
Total:  166 Filtered:  144
Accuracy: 0.8261, Precision: 0.8696, Recall: 0.4524, F1-score: 0.5951, Coverage: 0.5476


In [4]:
data = pd.read_csv('banknote.csv')

y = data.iloc[:, -1].to_numpy()
scaler = MinMaxScaler()
X = scaler.fit_transform(data.iloc[:, :-1])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

print('Maximal Cliques')
print('# of test:', len(y_test))

delta = 0.05 * math.sqrt(m)
alpha = 0.75

print('delta: ', delta, 'alpha: ', alpha)

# 使用向量化计算距离矩阵
dis_arr = pairwise_distances(X_train, metric='euclidean')
adj_arr = (dis_arr < delta).astype(int)

# 找到所有极大团
cliques = find_maximal_cliques(adj_arr)

print('Total: ', len(cliques))

# 并行过滤符合条件的团
def filter_clique(clique):
    temp_label = y_train[clique]
    count = pd.Series(temp_label).value_counts()
    if not count[count > alpha * len(clique)].empty:
        label = count[count > alpha * len(clique)].index[0]
        return clique, label
    else:
        return None

filtered_results = Parallel(n_jobs=-1)(delayed(filter_clique)(clique) for clique in cliques)

filtered_cliques = []
cliques_label = []

for result in filtered_results:
    if result is not None:
        clique, label = result
        filtered_cliques.append(clique)
        cliques_label.append(label)

print('Filtered: ', len(filtered_cliques))

acc = 0
dec = 0

# 初始化变量以收集预测结果和真实标签
y_true = []
y_pred = []

# 并行处理测试样本
def process_test_sample(i, t):
    mc_labels = []
    for k, clique in enumerate(filtered_cliques):
        # 检查测试样本与团中所有样本的距离是否都小于 delta
        distances = np.linalg.norm(X_train[clique] - t, axis=1)
        if np.all(distances < delta):
            mc_labels.append(cliques_label[k])
    if mc_labels:
        # 使用多数投票法确定预测标签
        predicted_label = pd.Series(mc_labels).mode()[0]
        return y_test[i], predicted_label
    else:
        return y_test[i], None  # 无法预测时，返回 None

test_results = Parallel(n_jobs=-1)(delayed(process_test_sample)(i, t) for i, t in enumerate(X_test))

# 收集预测结果和真实标签
for true_label, pred_label in test_results:
    y_true.append(true_label)
    y_pred.append(pred_label)

# 过滤掉无法预测的样本
y_true_filtered = []
y_pred_filtered = []
for true_label, pred_label in zip(y_true, y_pred):
    if pred_label is not None:
        y_true_filtered.append(true_label)
        y_pred_filtered.append(pred_label)

# 计算评价指标
if y_pred_filtered:
    precision = precision_score(y_true_filtered, y_pred_filtered, average='weighted', zero_division=0)
    recall = recall_score(y_true_filtered, y_pred_filtered, average='weighted', zero_division=0)
    f1 = f1_score(y_true_filtered, y_pred_filtered, average='weighted', zero_division=0)
    coverage = len(y_pred_filtered) / len(y_test)
    print(f'Precision: {precision:.4f}, Recall: {recall:.4f}, F1-score: {f1:.4f}, Coverage: {coverage:.4f}')
else:
    print('No predictions were made.')

Maximal Cliques
# of test: 275
delta:  0.3872983346207417 alpha:  0.75
Total:  4111585
Filtered:  1178892


exception calling callback for <Future at 0x27a93cd49a0 state=finished raised BrokenProcessPool>
joblib.externals.loky.process_executor._RemoteTraceback: 
"""
Traceback (most recent call last):
  File "C:\Users\Win_Nas\AppData\Local\Programs\Python\Python310\lib\site-packages\joblib\externals\loky\process_executor.py", line 391, in _process_worker
    call_item = call_queue.get(block=True, timeout=timeout)
  File "C:\Users\Win_Nas\AppData\Local\Programs\Python\Python310\lib\multiprocessing\queues.py", line 117, in get
    res = self._recv_bytes()
  File "C:\Users\Win_Nas\AppData\Local\Programs\Python\Python310\lib\multiprocessing\connection.py", line 221, in recv_bytes
    buf = self._recv_bytes(maxlength)
  File "C:\Users\Win_Nas\AppData\Local\Programs\Python\Python310\lib\multiprocessing\connection.py", line 323, in _recv_bytes
    return self._get_more_data(ov, maxsize)
  File "C:\Users\Win_Nas\AppData\Local\Programs\Python\Python310\lib\multiprocessing\connection.py", line 345, in 

BrokenProcessPool: A task has failed to un-serialize. Please ensure that the arguments of the function are all picklable.

In [100]:
print('# of test:', len(y_test))

delta = 0.15 * math.sqrt(m)
alpha = 0.7

print('delta: ', delta, 'alpha: ', alpha)

dis_arr = np.zeros((n,n))
neighborhoods = []
neighborhoods_label = []
granules = []

for i in range(n):
    neighbors = []
    temp_label = []
    for j in range(n):
        dis_arr[i][j] = minkowski_distance(X_train[i], X_train[j], 2)
        if dis_arr[i][j] < delta:
            neighbors.append(j)
            temp_label.append(y_train[j])
    
    count = pd.Series(temp_label).value_counts()
    if not count[count > alpha * len(neighbors)].empty:
        neighborhoods.append(i)
        granules.append(neighbors)
        neighborhoods_label.append(count[count > alpha * len(neighbors)].index.tolist()[0])

print('Total: ', n, 'Filtered: ', len(neighborhoods))
acc = 0
dec = 0

for i, t in enumerate(X_test):
    mc_labels = []
    for j in neighborhoods:
        ob = X_train[j]
        if minkowski_distance(t, ob, 2) < delta:
            mc_labels.append(y_train[j])
    if mc_labels:
        dec += 1
        percentage = pd.Series(mc_labels).value_counts(normalize=True) * 100
        res = ', '.join([f"{element}: {pct:.2f}%" for element, pct in percentage.items()])
        if mc_labels[0]==y_test[i]:
            acc += 1
    else:
        res = ''
    print(y_test[i], res)
print('qualit: ', acc/dec, dec/len(y_test))




acc = 0
dec = 0
h_Gs = []
for i, G in enumerate(granules):
    f_G ={}
    for x in G:
        count = sum(1 for k in G if minkowski_distance(X_train[k], X_train[x], 2) < delta)
        f_G[x] = count / len(G)
    total_f_G = sum(f_G.values())
    h_G = {x: f_G[x] / total_f_G for x in f_G}
    h_Gs.append(h_G)
    progress = (i+1) / len(granules) * 100
    #print(f"Progress: {progress:.2f}%", end='\r')
    
for i, t in enumerate(X_test):
    best_value = 0
    res = ''
    # for k, granule in enumerate(granules):
    #     phi_value = matching_degree(granule, h_Gs[k], t)
    #     if best_value < phi_value:
    #         best_value = phi_value
    #         res = neighborhoods_label[k]
    phi_values = [matching_degree(granule, h_Gs[k], t) for k, granule in enumerate(granules)]
    res = argmaxphi(phi_values, neighborhoods_label)
    if res:
        dec+=1
    if y_test[i]==res:
        acc+=1
    progress = (i+1) / len(X_test) * 100
    #print(f"Progress: {progress:.2f}%     ", end='\r')
    #print(y_test[i], res)

print(f"\nquanti: {(acc/dec*100):.2f}%", dec/len(y_test))

# of test: 42
delta:  1.161895003862225 alpha:  0.7
Total:  166 Filtered:  140
M M: 100.00%
R 
R R: 100.00%
R 
M 
R M: 83.33%, R: 16.67%
M 
M 
R R: 100.00%
M 
M 
R R: 100.00%
M M: 100.00%
M M: 100.00%
M M: 100.00%
M M: 90.91%, R: 9.09%
M 
M M: 90.00%, R: 10.00%
R R: 75.00%, M: 25.00%
R R: 100.00%
M M: 100.00%
M M: 100.00%
M 
M 
R R: 100.00%
R 
R 
R 
M 
M M: 100.00%
M R: 66.67%, M: 33.33%
R R: 100.00%
R R: 100.00%
M M: 90.91%, R: 9.09%
M 
M 
M 
M M: 100.00%
R 
M M: 87.50%, R: 12.50%
R R: 100.00%
M M: 75.00%, R: 25.00%
qualit:  0.9583333333333334 0.5714285714285714

quanti: 83.33% 0.5714285714285714


In [ ]:
print('# of test:', len(y_test))

delta = 0.2 * math.sqrt(m)
alpha = 0.85

print('delta: ', delta, 'alpha: ', alpha)

adj_arr = np.zeros((n,n))
dis_arr = np.zeros((n,n))

for i in range(n):
    for j in range(i, n):
        dis_arr[i][j] = minkowski_distance(X_train[i], X_train[j], 2)
        dis_arr[j][i] = dis_arr[i][j]
        if dis_arr[i][j] < delta:
            adj_arr[i][j] = 1
            adj_arr[j][i] = 1
            
cliques = find_maximal_cliques(adj_arr)

filtered_cliques = []
cliques_label = []
for clique in cliques:
    temp_label = []
    for j in clique:
        temp_label.append(y_train[j])
    
    count = pd.Series(temp_label).value_counts()
    if not count[count > alpha * len(clique)].empty:
        filtered_cliques.append(clique)
        cliques_label.append(count[count > alpha * len(clique)].index.tolist()[0])
        
print('Total: ', len(cliques), 'Filtered: ', len(filtered_cliques))
acc = 0
dec = 0

for i, t in enumerate(X_test):
    t_pos = 0;
    t_neg = 0;
    cy = []
    mc_labels = []
    for k, clique in enumerate(filtered_cliques):
        flag = True
        for j in clique:
            if minkowski_distance(t, X_train[j], 2) >= delta:
                flag = False
                break
        if flag:
            cy.append(clique)
            mc_labels.append(cliques_label[k])
    if mc_labels:
        dec += 1
        percentage = pd.Series(mc_labels).value_counts(normalize=True) * 100
        res = ', '.join([f"{element}: {pct:.2f}%" for element, pct in percentage.items()])
        if mc_labels[0]==y_test[i]:
            acc+=1
    else:
        res = ''
    #print(y_test[i], res)

print('qualit: ', acc/dec, dec/len(y_test))

acc = 0

h_Gs = []

for i, G in enumerate(filtered_cliques):
    f_G ={}
    for x in G:
        count = sum(1 for k in G if minkowski_distance(X_train[k], X_train[x], 2) < delta)
        f_G[x] = count / len(G)
    total_f_G = sum(f_G.values())
    h_G = {x: f_G[x] / total_f_G for x in f_G}
    h_Gs.append(h_G)
    progress = (i+1) / len(filtered_cliques) * 100
    print(f"Progress: {progress:.2f}%", end='\r')

for i, t in enumerate(X_test):
    best_value = 0
    res = ''
    # for k, clique in enumerate(filtered_cliques):
    #     phi_value = matching_degree(clique, h_Gs[k], t)
    #     if best_value < phi_value:
    #         best_value = phi_value
    #         res = cliques_label[k]
    
    phi_values = [matching_degree(granule, h_Gs[k], t) for k, granule in enumerate(filtered_cliques)]
    res = argmaxphi(phi_values, cliques_label)
    if res:
        dec+=1
    
    if y_test[i]==res:
        acc+=1
    progress = (i+1) / len(X_test) * 100
    print(f"Progress: {progress:.2f}%     ", end='\r')
    #print(y_test[i], res)

print(f"\nquanti: {(acc/dec*100):.2f}%", dec/len(y_test))

# of test: 114
delta:  1.0954451150103324 alpha:  0.85
Total:  493707 Filtered:  87444
